# AI vs Human Text Classification
We fine-tune BERT and ModernBERT models on the [AI vs Human Text](https://www.kaggle.com/datasets/shanegerami/ai-vs-human-text) dataset.

**Overview:**
- `model1`  – BERT, only the classification head is trained
- `model2` – BERT, full model fine-tuned
- `model3` – ModernBERT, only the classification head is trained
- `model4` – ModernBERT, full model fine-tuned

---
## Setup & Imports

In [ ]:
!pip install transformers opendatasets

In [ ]:
# Import packages
import os

import pandas as pd
import torch
from torch.utils.data import Dataset
from tqdm import tqdm
from sklearn import metrics
from transformers import (
    AutoTokenizer,
    BertForSequenceClassification,
    AutoModelForSequenceClassification,
)

# Select CPU or GPU
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

---
## Data

In [ ]:
# Download the dataset from Kaggle (enter Kaggle username and API key)
import opendatasets as od
od.download("https://www.kaggle.com/datasets/shanegerami/ai-vs-human-text")

# Rename the downloaded folder to "data"
original_folder = "ai-vs-human-text"
target_folder = "data"
if os.path.exists(original_folder) and not os.path.exists(target_folder):
    os.rename(original_folder, target_folder)
    print(f"Renamed '{original_folder}' to '{target_folder}'")

In [ ]:
# Load the CSV and inspect the dataset size
df = pd.read_csv("data/AI_Human.csv")
print(f"Total samples: {len(df)}")
print(f"Proportion of AI-generated texts: {df['generated'].mean():.2f}")

In [ ]:
# Split into train / validation / test  (70 / 15 / 15)
df_train_temp = df.sample(frac=0.85, random_state=5)
df_test = df.drop(df_train_temp.index)

df_valid = df_train_temp.sample(frac=0.15, random_state=5)
df_train = df_train_temp.drop(df_valid.index)

print(f"Train: {len(df_train):,}  |  Validation: {len(df_valid):,}  |  Test: {len(df_test):,}")

---
## Dataset Classes
 
Padding to `max_length=512`.

In [4]:
class AIDataset_Bert(Dataset):
    """PyTorch Dataset that tokenises text with the BERT tokeniser."""

    def __init__(self, df):
        self.df = df
        self.tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        tokenized = self.tokenizer(
            row["text"],
            padding="max_length",
            max_length=512,   # BERT maximum sequence length
            truncation=True,
            return_tensors="pt",
        )
        # Cast label to long (required by CrossEntropyLoss)
        tokenized["label"] = torch.tensor(row["generated"], dtype=torch.long)
        return tokenized


class AIDataset_Modern(Dataset):
    """PyTorch Dataset that tokenises text with the ModernBERT tokeniser."""

    def __init__(self, df):
        self.df = df
        self.tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        tokenized = self.tokenizer(
            row["text"],
            padding="max_length",
            max_length=512,
            truncation=True,
            return_tensors="pt",
        )
        tokenized["label"] = torch.tensor(row["generated"], dtype=torch.long)
        return tokenized

---
## Functions for Inference

In [5]:
def predict(model, df_valid, valid_loader, squeeze_attention_mask=False):
    """
    Run inference on "valid_loader" and attach predictions/probabilities to "df_valid".

    Args:
        model: A HuggingFace sequence-classification model.
        df_valid: The DataFrame corresponding to `valid_loader`.
        valid_loader: DataLoader for the validation/test set.
        squeeze_attention_mask: Set True for ModernBERT, which requires a 2-D mask.

    Returns:
        df_valid with new columns: "preds" and "probs".
    """
    probs_list = []

    # Disable dropout for evaluation
    model.eval()
    with torch.no_grad():
        for batch in tqdm(valid_loader):
            # input_ids shape: [batch, 1, 512] -> squeeze to [batch, 512]
            input_ids = batch["input_ids"].to(device).squeeze(dim=1)
            attention_mask = batch["attention_mask"].to(device)
            if squeeze_attention_mask:
                # ModernBERT expects 2-D attention mask
                attention_mask = attention_mask.squeeze(dim=1)

            outputs = model(input_ids, attention_mask=attention_mask)
            probs_list += torch.nn.functional.softmax(outputs["logits"], dim=1)

    preds = [torch.argmax(p, dim=0).item() for p in probs_list]
    df_valid = df_valid.copy()
    df_valid["preds"] = preds
    df_valid["probs"] = [p.cpu().tolist() for p in probs_list]
    return df_valid


def evaluate(df_res, label_col="generated"):
    """Print accuracy, confusion matrix, and a full classification report."""
    accuracy = (df_res[label_col] == df_res["preds"]).mean()
    print(f"Accuracy: {accuracy:.4f}")
    print(
        f"\033[94m Confusion Matrix: \033[0m \n"
        f"{pd.crosstab(df_res[label_col], df_res['preds'], margins=True)}"
    )
    print(metrics.classification_report(y_true=df_res[label_col], y_pred=df_res["preds"], digits=4))


def make_optimizer_and_scheduler(model, train_loader, nr_epochs, lr=1e-5):
    """Create the AdamW optimizer, CrossEntropyLoss, and a OneCycleLR scheduler."""
    optimizer      = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_function  = torch.nn.CrossEntropyLoss()
    scheduler      = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        epochs=nr_epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,   # 10 % of steps are warm-up
    )
    return optimizer, loss_function, scheduler


def train_one_run(model, train_loader, optimizer, loss_function, scheduler,
                  nr_epochs, squeeze_attention_mask=False, log_every=250):
    """
    Full training loop for "nr_epochs".

    Args:
        model: Model to train (must already be in train mode and on "device").
        train_loader: DataLoader for the training set.
        optimizer / loss_function / scheduler: from "make_optimizer_and_scheduler".
        nr_epochs: Number of full passes over the training set.
        squeeze_attention_mask: Set True for ModernBERT.
        log_every: Print running loss every this many iterations.
    """
    loss_logger = []

    for epoch in range(nr_epochs):
        print(f"Epoch {epoch + 1} / {nr_epochs} started")
        running_loss = 0.0

        for i, batch in enumerate(tqdm(train_loader)):
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device).squeeze(dim=1)
            attention_mask = batch["attention_mask"].to(device)
            if squeeze_attention_mask:
                attention_mask = attention_mask.squeeze(dim=1)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = loss_function(outputs["logits"], labels)

            running_loss += loss.item()

            # Log average loss every "log_every" iterations
            if i % log_every == (log_every - 1):
                avg_loss = running_loss / log_every
                lr_now   = scheduler.get_last_lr()
                print(f" iter: {i+1}, loss: {avg_loss:.8f}, lr: {lr_now}")
                loss_logger.append({"iter": i + 1, "loss": avg_loss})
                running_loss = 0.0

            # Gradient update
            loss.backward()
            optimizer.step()
            scheduler.step()

    return loss_logger

---
## BERT Models

### Load models & build data loaders

In [ ]:
BERT_CHECKPOINT = "google-bert/bert-base-uncased"

model1 = BertForSequenceClassification.from_pretrained(BERT_CHECKPOINT, num_labels=2)
model2 = BertForSequenceClassification.from_pretrained(BERT_CHECKPOINT, num_labels=2)

# Move both models to the selected device and put in training mode
# (training mode keeps dropout active, eval mode disables it)
model1.to(device).train()
model2.to(device).train()
print("BERT models loaded and in training mode")

In [ ]:
# Build datasets and dataloaders for BERT
trainBert_dataset = AIDataset_Bert(df=df_train)
validBert_dataset = AIDataset_Bert(df=df_valid)
testBert_dataset  = AIDataset_Bert(df=df_test)

# Inspect one tokenised sample
sample = trainBert_dataset[0]
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_CHECKPOINT)
print("Decoded tokens (first sample):", bert_tokenizer.convert_ids_to_tokens(sample["input_ids"][0])[:20])

trainBert_loader = torch.utils.data.DataLoader(trainBert_dataset, batch_size=64, shuffle=True,  num_workers=2)
validBert_loader = torch.utils.data.DataLoader(validBert_dataset, batch_size=64, shuffle=False, num_workers=2)
testBert_loader = torch.utils.data.DataLoader(testBert_dataset,  batch_size=64, shuffle=False, num_workers=2)

### Model 1 (BERT classification head only)

Freeze all BERT encoder layers. Only the pooler and classifier are updated. 1 epoch.

In [ ]:
# Freeze all parameters
for param in model1.parameters():
    param.requires_grad = False

# Unfreeze the head
model1.bert.pooler.dense.weight.requires_grad = True
model1.bert.pooler.dense.bias.requires_grad = True
model1.classifier.weight.requires_grad = True
model1.classifier.bias.requires_grad = True

# Print trainable parameter names
# n = name of parameter; p = parameter object (weights)
trainable = [n for n, p in model1.named_parameters() if p.requires_grad]
print("Trainable parameters:", trainable)

In [ ]:
# Training setup (1 epoch)
optimizer_m, loss_fn_m, scheduler_m = make_optimizer_and_scheduler(model1, trainBert_loader, nr_epochs=1)

# Run training
train_one_run(
    model1, trainBert_loader, optimizer_m, loss_fn_m, scheduler_m,
    nr_epochs=1, squeeze_attention_mask=False, log_every=250
)

In [ ]:
# Evaluate on the validation set
df_res_m = predict(model1, df_valid, validBert_loader, squeeze_attention_mask=False)
evaluate(df_res_m)

### Model 2 (Full BERT fine-tuning)

In [ ]:
# All parameters are trainable by default
# Training setup (2 epochs, same LR)
optimizer_m2, loss_fn_m2, scheduler_m2 = make_optimizer_and_scheduler(model2, trainBert_loader, nr_epochs=2)

train_one_run(
    model2, trainBert_loader, optimizer_m2, loss_fn_m2, scheduler_m2,
    nr_epochs=2, squeeze_attention_mask=False, log_every=500
)

In [ ]:
# Evaluate on the validation set
df_res_m2 = predict(model2, df_valid, validBert_loader, squeeze_attention_mask=False)
evaluate(df_res_m2)

In [ ]:
# Save model2
import os
from google.colab import files

output_dir = "./my_model2"
os.makedirs(output_dir, exist_ok=True)
model2.save_pretrained(output_dir)
bert_tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

!zip -r /content/my_model2.zip /content/my_model2
files.download("/content/my_model2.zip")

---
## ModernBERT models

### Load models & build data loaders

In [ ]:
from transformers import AutoConfig, AutoModelForSequenceClassification

MODERN_CHECKPOINT = "answerdotai/ModernBERT-base"

model3 = AutoModelForSequenceClassification.from_pretrained(MODERN_CHECKPOINT, num_labels=2, trust_remote_code=True)
model4 = AutoModelForSequenceClassification.from_pretrained(MODERN_CHECKPOINT, num_labels=2, trust_remote_code=True)

model3.to(device).train()
model4.to(device).train()
print("ModernBERT models loaded and in training mode")

In [ ]:
# Datasets and dataloaders for ModernBERT
train_dataset = AIDataset_Modern(df=df_train)
valid_dataset = AIDataset_Modern(df=df_valid)
test_dataset = AIDataset_Modern(df=df_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=64, shuffle=False, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2)

### Model 3 (ModernBERT classification head only)

In [ ]:
# Freeze all parameters
for param in model3.parameters():
    param.requires_grad = False

# Unfreeze head + classifier
model3.head.dense.weight.requires_grad  = True
model3.head.norm.weight.requires_grad   = True
model3.classifier.weight.requires_grad  = True
model3.classifier.bias.requires_grad    = True

trainable3 = [n for n, p in model3.named_parameters() if p.requires_grad]
print("Trainable parameters:", trainable3)

In [ ]:
# Training setup. 1 epoch
optimizer_m3, loss_fn_m3, scheduler_m3 = make_optimizer_and_scheduler(model3, train_loader, nr_epochs=1)

# Run training
train_one_run(
    model3, train_loader, optimizer_m3, loss_fn_m3, scheduler_m3,
    nr_epochs=1, squeeze_attention_mask=True, log_every=250
)

In [ ]:
# Evaluate on the validation set
df_res_m3 = predict(model3, df_valid, valid_loader, squeeze_attention_mask=True)
evaluate(df_res_m3)

In [ ]:
# Save model3
modern_tokenizer = AutoTokenizer.from_pretrained(MODERN_CHECKPOINT)

output_dir3 = "./my_model3"
os.makedirs(output_dir3, exist_ok=True)
model3.save_pretrained(output_dir3)
modern_tokenizer.save_pretrained(output_dir3)
print(f"Model saved to {output_dir3}")

!zip -r /content/my_model3.zip /content/my_model3
files.download("/content/my_model3.zip")

### Model 4 (Full ModernBERT fine-tuning)

In [ ]:
# Training setup
optimizer_m4, loss_fn_m4, scheduler_m4 = make_optimizer_and_scheduler(model4, train_loader, nr_epochs=1)

# Run training
train_one_run(
    model4, train_loader, optimizer_m4, loss_fn_m4, scheduler_m4,
    nr_epochs=1, squeeze_attention_mask=True, log_every=250
)

In [ ]:
# Evaluate on the validation set
df_res_m4 = predict(model4, df_valid, valid_loader, squeeze_attention_mask=True)
evaluate(df_res_m4)

In [ ]:
# Save model4
output_dir4 = "./my_model4"
os.makedirs(output_dir4, exist_ok=True)
model4.save_pretrained(output_dir4)
modern_tokenizer.save_pretrained(output_dir4)
print(f"Model saved to {output_dir4}")

!zip -r /content/my_model4.zip /content/my_model4
files.download("/content/my_model4.zip")

---
## Final Test-set Evaluation

Evaluate all four models on the held-out test set.

In [ ]:
for name, mdl, loader, squeeze in [
    ("model  (BERT head-only)", model, testBert_loader, False),
    ("model2 (BERT full fine-tune)", model2, testBert_loader, False),
    ("model3 (ModernBERT head-only)", model3, test_loader, True),
    ("model4 (ModernBERT full ft)", model4, test_loader, True),
]:
    print(f"\n{'='*60}\n{name}")
    df_test_res = predict(mdl, df_test, loader, squeeze_attention_mask=squeeze)
    evaluate(df_test_res)

---
## Inference on Custom Texts

In [ ]:
# Mount Google Drive
from google.colab import drive
os.makedirs("content", exist_ok=True)
drive.mount("content/drive")

In [ ]:
# Unzip saved model checkpoints from Drive
!unzip /content/content/drive/MyDrive/ColabNotebooks/my_model2.zip
!unzip /content/content/drive/MyDrive/ColabNotebooks/my_model3.zip
!unzip /content/content/drive/MyDrive/ColabNotebooks/my_model4.zip

In [ ]:
# Reload models from the unzipped checkpoints
model2_inf = BertForSequenceClassification.from_pretrained("content/my_model2").to(device)
model3_inf = AutoModelForSequenceClassification.from_pretrained("content/my_model3").to(device)
model4_inf = AutoModelForSequenceClassification.from_pretrained("content/my_model4").to(device)
print("Models loaded successfully")

In [ ]:
# Load the inference dataset
df_inf = pd.read_csv("/content/content/drive/MyDrive/ColabNotebooks/model_inference.txt")
print(df_inf)

In [ ]:
# Build inference dataloaders
inf_dataset_bert   = AIDataset_Bert(df=df_inf)
inf_dataset_modern = AIDataset_Modern(df=df_inf)

inf_loader_bert   = torch.utils.data.DataLoader(inf_dataset_bert,   batch_size=1, shuffle=False)
inf_loader_modern = torch.utils.data.DataLoader(inf_dataset_modern, batch_size=1, shuffle=False)

In [ ]:
# Run inference with fine-tuned BERT and ModernBERT
df_inf_bert   = predict(model2_inf, df_inf, inf_loader_bert,   squeeze_attention_mask=False)
df_inf_modern = predict(model4_inf, df_inf, inf_loader_modern, squeeze_attention_mask=True)

print("\n--- BERT (full fine-tune) ---")
evaluate(df_inf_bert)

print("\n--- ModernBERT (full fine-tune) ---")
evaluate(df_inf_modern)